In [ ]:
def main(datasources, start_date, end_date):
    """
    pv_frag_shock · 碎单超额冲击 → 次日回吐

    1) 谁付钱: 散户/碎单流——相对自身历史，笔数扩张快于成交量，同样活跃被切碎推价
    2) 次日: H_cont（实证定号）；factor = CSRank(raw)
    3) 为何必须 deal_number: 只用 volume/amount → 退化为 vol_shock_on；本因子身份是「谁在交易」

    结构（从链条倒推，单层）:
      raw = log(deal / MA60(deal).shift(1)) − log(vol / MA60(vol).shift(1))
      即 log(deal/vol) 相对自身 60 日基线的偏离；raw>0 ⇒ 今天比历史更碎
    窗: D=60 满窗（对齐 vol_shock_on 冲击基线）；min_sample=30
    LOOKBACK: 60*3+40；池左连，不 fillna(0)
    """
    import pandas as pd
    import numpy as np
    import dai

    bar1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")
    D, MIN_SAMPLE = 60, 30
    LOOKBACK_DAYS = D * 3 + 40
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)

    def _cs_rank(s):
        if int(s.notna().sum()) < MIN_SAMPLE:
            return pd.Series(np.nan, index=s.index)
        return s.rank(pct=True)

    sql = f"""
    SELECT
        strftime(date_trunc('day', date), '%Y-%m-%d') AS trading_day,
        instrument,
        SUM(volume) AS volume,
        SUM(deal_number) AS deal_number
    FROM {bar1m}
    WHERE volume > 0 AND deal_number > 0 AND close > 0
    GROUP BY date_trunc('day', date), instrument
    """
    px = dai.query(sql, filters={"date": [query_start_date, end_date]}, compression=True).df()
    for c in ("volume", "deal_number"):
        px[c] = pd.to_numeric(px[c], errors="coerce")
    px["date"] = pd.to_datetime(px["trading_day"]).dt.normalize()
    px["instrument"] = px["instrument"].astype(str)
    px = px.sort_values(["instrument", "date"]).reset_index(drop=True)

    g = px.groupby("instrument", sort=False)
    deal_hat = g["deal_number"].transform(lambda s: s.rolling(D, min_periods=D).mean().shift(1))
    vol_hat = g["volume"].transform(lambda s: s.rolling(D, min_periods=D).mean().shift(1))
    px["raw"] = np.log(px["deal_number"] / deal_hat.replace(0, np.nan)) - np.log(
        px["volume"] / vol_hat.replace(0, np.nan)
    )
    px["factor"] = px.groupby("date")["raw"].transform(_cs_rank)

    sd, ed = pd.to_datetime(start_date), pd.to_datetime(end_date)
    fac = px.loc[(px["date"] >= sd) & (px["date"] <= ed), ["date", "instrument", "factor"]]
    fac["factor"] = pd.to_numeric(fac["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    fac = fac.drop_duplicates(["date", "instrument"], keep="last")

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
    pool["instrument"] = pool["instrument"].astype(str)
    out = pool.merge(fac, how="left", on=["date", "instrument"])
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    return out[["date", "instrument", "factor"]].reset_index(drop=True)


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date = "2023-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    logger.info(f"因子行数={len(factor_data)} na={factor_data['factor'].isna().mean():.3f}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data, factor_pool=factor_pool, process_pools=False, show=True,
        start_date="2023-01-01", end_date="2024-12-31",
    )


[2026-07-23 10:35:56] [info     ] 计算因子，区间：2023-01-01 00:00:00 ~ 2024-12-31 23:59:59
